In [15]:
import os
import numpy as np
import pandas as pd
from nn_jax_ver.data_process import *
from nn_jax_ver.trainer import *
from nn_jax_ver.nn_model import *

from nn_jax_ver.fitness import *
from nn_jax_ver.dist import *
from nn_jax_ver.selection import *

In [4]:
jax.devices()

[CpuDevice(id=0)]

# data

In [2]:
# -------- load wine dataset (local first) --------
RED_PATH = "winequality-red.csv"
WHITE_PATH = "winequality-white.csv"

if os.path.exists(RED_PATH) and os.path.exists(WHITE_PATH):
    print("✅ load from local files")
    red = pd.read_csv(RED_PATH, sep=";")
    white = pd.read_csv(WHITE_PATH, sep=";")
else:
    print("⬇️ download from UCI")
    red = pd.read_csv(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv",
        sep=";"
    )
    white = pd.read_csv(
        "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv",
        sep=";"
    )

red["type"] = "red"
white["type"] = "white"

df = pd.concat([red, white], ignore_index=True)

print(df.shape)
print(df.head())


⬇️ download from UCI
(6497, 13)
   fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0            7.4              0.70         0.00             1.9      0.076   
1            7.8              0.88         0.00             2.6      0.098   
2            7.8              0.76         0.04             2.3      0.092   
3           11.2              0.28         0.56             1.9      0.075   
4            7.4              0.70         0.00             1.9      0.076   

   free sulfur dioxide  total sulfur dioxide  density    pH  sulphates  \
0                 11.0                  34.0   0.9978  3.51       0.56   
1                 25.0                  67.0   0.9968  3.20       0.68   
2                 15.0                  54.0   0.9970  3.26       0.65   
3                 17.0                  60.0   0.9980  3.16       0.58   
4                 11.0                  34.0   0.9978  3.51       0.56   

   alcohol  quality type  
0      9.4        5  red  


In [3]:
dm=WineNNDataManager(df)
data = dm.get_numpy()

In [5]:
key = jax.random.PRNGKey(0)
key, subkey = jax.random.split(key)
m1key,m2key,m3key,m4key=jrnd.split(key,4)

In [6]:
encoder=navie_nn(
    key=m1key,
    dim=12,
    hidden_dims=[32, 32],
    out_dim=3,
    cdim=3,
)
decoder=decoding(key=m2key,
                 dg=32,
                 dim=3,
                 dk=16,
                 dout=14)
print_model_stats(encoder)
print_model_stats(decoder)

=== Model ===
Parameters : 1,886
Param size : 0.007 MiB
=== Model ===
Parameters : 4,510
Param size : 0.017 MiB


In [7]:
optimizer = optax.adam(1e-3)

# Filter the model to get only array parameters for optax initialization
params = eqx.filter(encoder, eqx.is_array)
opt_state = optimizer.init(params)

# run
encoder, opt_state = encoding_train_runner(
    encoder,
    data,
    optimizer,
    opt_state,
    lambda1=0.5,
    lambda2=0.5,
    epochs=1,
)

[epoch 0] avg loss = 3.9228439331054688


In [8]:
optimizer = optax.adam(1e-3)

params = eqx.filter(decoder, eqx.is_array)
opt_state = optimizer.init(params)

decoder, opt_state = decoding_train_runner(
    
    encoder,
    decoder,
    data,
    optimizer,
    opt_state,
    epochs=1)

[ep 0 | it 0] loss=8486.2686 score=8130.4287 kl=355.8396
[ep 0 | it 10] loss=2682.2854 score=2530.8931 kl=151.3924
[ep 0 | it 20] loss=935.1234 score=882.3765 kl=52.7468
[ep 0 | it 30] loss=282.4431 score=267.8504 kl=14.5927
[ep 0 | it 40] loss=83.4954 score=80.0019 kl=3.4935
[ep 0 | it 50] loss=38.7649 score=37.4361 kl=1.3288
[ep 0 | it 60] loss=16.7121 score=16.4023 kl=0.3098
[ep 0 | it 70] loss=9.5139 score=9.4237 kl=0.0902
[ep 0 | it 80] loss=5.5119 score=5.4711 kl=0.0408
[ep 0 | it 90] loss=4.9570 score=4.9192 kl=0.0378
[ep 0 | it 100] loss=3.0321 score=3.0362 kl=-0.0041
[ep 0 | it 110] loss=2.2323 score=2.2461 kl=-0.0138
[ep 0 | it 120] loss=2.7702 score=2.7808 kl=-0.0106
[ep 0 | it 130] loss=1.7460 score=1.8088 kl=-0.0628
[ep 0 | it 140] loss=4.3010 score=4.3158 kl=-0.0148
[ep 0 | it 150] loss=3.1392 score=3.1604 kl=-0.0213
[ep 0 | it 160] loss=3.3667 score=3.3525 kl=0.0142
[ep 0 | it 170] loss=4.0996 score=4.1283 kl=-0.0286
[ep 0 | it 180] loss=1.8650 score=1.8675 kl=-0.0025
[e

In [9]:
batch = get_batch(data, subkey, batch_size=32)

In [10]:
g1=build_type_graph(batch['type'])
g2=build_type_graph(batch['quality'])
g=jnp.stack([g1,g2],axis=0)


In [11]:
m=encoder(batch['x'],g)

reject=cov_mu_reject(m,m3key)

In [12]:
reject

Array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1], dtype=int32)

In [13]:
muo=jnp.ones(m.shape[0])/m.shape[0]
covo=jnp.diag(muo*(1-muo))
score_hat=compute_decoder_score(m,decoder)
mu,cov=update_prior_gaussian(muo,covo,m,score_hat)
llr=lrt_ni_vs_n(mu,cov)
chi_stat = 2 * llr

reject = (chi_stat > 3.84)

In [14]:
reject

Array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True], dtype=bool)